# Lanugage Embedding - Model Preparation

While I have implemented recommended settings in the Model (MixedPrecision, GradientAccumulation, Using GPU, ...) to increase the training speed, none of them had an effect. However, replacing the sentence embedding model with a static version (as recomended by the library to increase performance) has masivly increased the speed from over 1 day per loop to 2 hours per loop (ona  single CPU). Unfortunatly, even when utilizing the onnx backend for the sentence embedder, the speed does not increase further. Using a GPU through Google Colab does not only not even improve the speed, but has a worse performance at 2,5 hours per loop. As the dataloader with the sentence embedding seems to be my current bottleneck, I have decided to attempt to pre-embedd all the products and queries beforehand and save the embedded vectors in a dataset.

In [1]:
import torch
import os

import numpy as np
import pandas as pd

device = torch.device("cpu")
accelerator = "cpu"
data_dir = "esci-data/shopping_queries_dataset"

Products = pd.read_parquet(f"{data_dir}/shopping_queries_dataset_products.parquet").fillna("")
Products.loc[Products.product_color == "Negro (", "product_color"] = "Negro"
Examples = pd.read_parquet(f"{data_dir}/shopping_queries_dataset_examples.parquet").fillna("")

In [2]:
Dataset = pd.merge(Products, Examples, on = ["product_id", "product_locale"]).drop_duplicates()
Dataset.head()

,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale,example_id,query,query_id,esci_label,small_version,large_version,split
0,B079VKKJN7,"11 Degrees de los Hombres Playera con Logo, Ne...",Esta playera con el logo de la marca Carrier d...,11 Degrees Negro Playera con logo\nA estrenar ...,11 Degrees,Negro,es,47092,11 degrees,1688,E,0,1,train
1,B079Y9VRKS,Camiseta Eleven Degrees Core TS White (M),,,11 Degrees,Blanco,es,47091,11 degrees,1688,E,0,1,train
2,B07DP4LM9H,11 Degrees de los Hombres Core Pull Over Hoodi...,La sudadera con capucha Core Pull Over de 11 G...,11 Degrees Azul Core Pull Over Hoodie\nA estre...,11 Degrees,Azul,es,47089,11 degrees,1688,E,0,1,train
3,B07G37B9HP,11 Degrees Poli Panel Track Pant XL Black,,,11 Degrees,,es,47088,11 degrees,1688,E,0,1,train
4,B07LCTGDHY,11 Degrees Gorra Trucker Negro OSFA (Talla úni...,,,11 Degrees,Negro,es,47087,11 degrees,1688,E,0,1,train


In [3]:
Large_Dataset = Dataset.loc[Dataset.large_version == 1, :].drop(columns = ["small_version", "large_version"])
Large_Dataset.shape

(2621288, 12)

In [4]:
del Dataset
del Products
del Examples

In [5]:
from sentence_transformers import SentenceTransformer
# "distiluse-base-multilingual-cased-v2"
# The static model has less quality in its performance compared to the distilusse, but in turn offers much faster computation speeds. Tiem fro one Epoch with CPU: From 1day8h to merely 2h just by switching to the static model.

model_kwargs = {"file_name": "onnx/model.onnx"}

embedding_model = SentenceTransformer("sentence-transformers/static-similarity-mrl-multilingual-v1", backend="onnx", model_kwargs = model_kwargs) # ONNX backend is supposed to help with speed

In [6]:
columns_for_embedding = ["product_title", "product_description", "product_bullet_point", "product_brand", "product_color", "product_locale", "query"]
columns_with_info = ["product_id", "query_id", "example_id", "split", "esci_label"]
Large_Dataset = Large_Dataset.loc[:, tuple(columns_for_embedding + columns_with_info)]
Large_Dataset

,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale,query,product_id,query_id,example_id,split,esci_label
0,"11 Degrees de los Hombres Playera con Logo, Ne...",Esta playera con el logo de la marca Carrier d...,11 Degrees Negro Playera con logo\nA estrenar ...,11 Degrees,Negro,es,11 degrees,B079VKKJN7,1688,47092,train,E
1,Camiseta Eleven Degrees Core TS White (M),,,11 Degrees,Blanco,es,11 degrees,B079Y9VRKS,1688,47091,train,E
2,11 Degrees de los Hombres Core Pull Over Hoodi...,La sudadera con capucha Core Pull Over de 11 G...,11 Degrees Azul Core Pull Over Hoodie\nA estre...,11 Degrees,Azul,es,11 degrees,B07DP4LM9H,1688,47089,train,E
3,11 Degrees Poli Panel Track Pant XL Black,,,11 Degrees,,es,11 degrees,B07G37B9HP,1688,47088,train,E
4,11 Degrees Gorra Trucker Negro OSFA (Talla úni...,,,11 Degrees,Negro,es,11 degrees,B07LCTGDHY,1688,47087,train,E
...,...,...,...,...,...,...,...,...,...,...,...,...
2621283,レディース ブラジャー 補正ブラ トップス Gabrioir 大きいサイズ 揺れない 脇高設...,<p>【配送】</p><p>普通には2-3日後出荷できますが。出荷した後15-25日ぐらい届...,【いろんな場合に適用】軽い運動時、家事、外出、長時間移動、温泉旅行などに活用なセクシルームブ...,Gabrioir,ベージュ,jp,肩なしブラジャー,B09G72LDQZ,129052,2580177,train,S
2621284,ナイトブラ ノンワイヤーブラ ブラジャー レディース Gabrioir 大きいサイズ パット...,<p>【配送】</p><p>普通には2-3日後出荷できますが。出荷した後15-25日ぐらい届...,【いろんな場合に適用】軽い運動時、家事、外出、長時間移動、温泉旅行などに活用なセクシルームブ...,Gabrioir,パープル,jp,肩なしブラジャー,B09GBGRTPB,129052,2580178,train,S
2621285,インナーキャミソール レディース ブラトップ キャミソール Gabrioir ノーワイヤー ...,<p>【配送】</p><p>普通には2-3日後出荷できますが。出荷した後15-25日ぐらい届...,【いろんな場合に適用】軽い運動時、家事、外出、長時間移動、温泉旅行などに活用なセクシルームブ...,Gabrioir,ブラック,jp,ベアトップ カップなし,B09GBKN83Q,123197,2433083,train,S
2621286,インナーキャミソール レディース ブラトップ キャミソール Gabrioir ノーワイヤー ...,<p>【配送】</p><p>普通には2-3日後出荷できますが。出荷した後15-25日ぐらい届...,【いろんな場合に適用】軽い運動時、家事、外出、長時間移動、温泉旅行などに活用なセクシルームブ...,Gabrioir,ブラック,jp,女性 チューブトップ,B09GBKN83Q,126351,2509401,train,E


In [7]:
batch_size = 32
tensor_size = 32
embedding_func = lambda x: embedding_model.encode(x,  output_value = "sentence_embedding", truncate_dim = tensor_size, convert_to_tensor = False, batch_size = batch_size * 4, show_progress_bar = True)

In [8]:
multi_column_prep = []
for column_name in columns_for_embedding:
    for number in range(tensor_size):
        multi_column_prep.append((column_name, number))
multi_column_index = pd.MultiIndex.from_tuples(multi_column_prep)

In [9]:
Embedded_df = embedding_func(Large_Dataset.loc[:, columns_for_embedding[0]])
for i in range(1, len(columns_for_embedding)):
    additional = embedding_func(Large_Dataset.loc[:, columns_for_embedding[i]])
    Embedded_df = np.concat((Embedded_df, additional), axis = 1)



Batches:   0%|          | 0/20479 [00:00<?, ?it/s]

Batches:   0%|          | 0/20479 [00:00<?, ?it/s]

Batches:   0%|          | 0/20479 [00:00<?, ?it/s]

Batches:   0%|          | 0/20479 [00:00<?, ?it/s]

Batches:   0%|          | 0/20479 [00:00<?, ?it/s]

Batches:   0%|          | 0/20479 [00:00<?, ?it/s]

Batches:   0%|          | 0/20479 [00:00<?, ?it/s]

In [10]:
Embedded_df = pd.DataFrame(Embedded_df, columns = multi_column_index)
for col in columns_with_info:
    Embedded_df.loc[:, col] = Large_Dataset.loc[:, col]

In [11]:
Embedded_df

product_title                                                       \
                    0         1         2          3          4          5   
0           -4.079337  3.847864 -0.729618   3.138518   4.327556   3.357860   
1            3.893338  5.651582  2.524054   1.534307   1.740349   4.919367   
2           -1.541733  5.915594  2.462360   5.830202   0.057655   3.485694   
3            7.090992  7.519460 -1.712093  14.279944 -10.086328  12.536506   
4            6.522617  5.568812 -1.813076   8.262443   4.007835   1.429445   
...               ...       ...       ...        ...        ...        ...   
2621283      4.953020  1.788919  1.748300   0.622110  -3.231584   1.764582   
2621284      5.069838  1.883011 -0.424063  -2.065871  -4.235208   0.511501   
2621285      6.003750  0.395459  1.298870  -0.893088  -4.217788   0.959003   
2621286      6.003750  0.395459  1.298870  -0.893088  -4.217788   0.959003   
2621287      6.003750  0.395459  1.298870  -0.893088  -4.217788   0.959003   

                                                 ...      query             \
                6         7         8         9  ...         27         28   
0        1.216648  3.532077 -0.723933 -0.337356  ... -12.513047  15.264223   
1       -1.271370  2.135581  0.796760  1.958821  ... -12.513047  15.264223   
2       -1.508375  6.050313  2.976396  1.494216  ... -12.513047  15.264223   
3       -0.105441 -0.776574 -0.962799 -1.286689  ... -12.513047  15.264223   
4        2.572388  4.718240  1.622925  1.972882  ... -12.513047  15.264223   
...           ...       ...       ...       ...  ...        ...        ...   
2621283 -2.250641 -3.608654  0.817755  1.967357  ...  10.972528  -0.366776   
2621284  0.463555 -0.206944 -0.565094  0.898649  ...  10.972528  -0.366776   
2621285  2.384049 -2.821439  1.133335  1.409757  ...   6.753409   6.227701   
2621286  2.384049 -2.821439  1.133335  1.409757  ...  10.872013 -10.514281   
2621287  2.384049 -2.821439  1.133335  1.409757  ...  10.972528  -0.366776   

                                         product_id query_id example_id  \
                29         30        31                                   
0       -19.144426 -22.956654  7.925159  B079VKKJN7     1688      47092   
1       -19.144426 -22.956654  7.925159  B079Y9VRKS     1688      47091   
2       -19.144426 -22.956654  7.925159  B07DP4LM9H     1688      47089   
3       -19.144426 -22.956654  7.925159  B07G37B9HP     1688      47088   
4       -19.144426 -22.956654  7.925159  B07LCTGDHY     1688      47087   
...            ...        ...       ...         ...      ...        ...   
2621283  -5.341568   9.457088  3.555554  B09G72LDQZ   129052    2580177   
2621284  -5.341568   9.457088  3.555554  B09GBGRTPB   129052    2580178   
2621285   0.799776   0.190105  4.898800  B09GBKN83Q   123197    2433083   
2621286   1.899601   7.058802  0.403267  B09GBKN83Q   126351    2509401   
2621287  -5.341568   9.457088  3.555554  B09GBKN83Q   129052    2580179   

         split esci_label  
                           
0        train          E  
1        train          E  
2        train          E  
3        train          E  
4        train          E  
...        ...        ...  
2621283  train          S  
2621284  train          S  
2621285  train          S  
2621286  train          E  
2621287  train          S  

[2621288 rows x 229 columns]

In [12]:
Embedded_df.to_parquet("EmbeddedData/SentenceEmbedding.parquet")

C:\Users\flash\anaconda3\envs\ML_Pytorch\Lib\site-packages\pandas\io\parquet.py:191: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  table = self.api.Table.from_pandas(df, **from_pandas_kwargs)
